# Predicting Customer Churn Using Machine Learning
**DSC 680 — Project 3, Milestone 3**  
**Student:** Abdelaziz Frédéric Ouattara

## Abstract
Customer churn prediction can help organizations identify customers at elevated risk of leaving and prioritize retention resources. This project analyzes the IBM Telco Customer Churn dataset, cleans and prepares customer demographic, service, and account variables, explores churn patterns, and compares Logistic Regression, Decision Tree, and Random Forest classifiers. Evaluation emphasizes not only accuracy but also precision, recall, F1-score, ROC-AUC, cross-validation, confusion matrices, and interpretability. Ethical considerations include privacy, proxy discrimination, unequal error rates, transparency, and the risk of using predictions to unfairly target customers.

## Research Questions
1. Which customer characteristics are most strongly associated with customer churn?
2. Can machine learning accurately predict whether a customer will leave?
3. Which classification algorithm performs best for predicting customer churn?
4. What business recommendations can be made to reduce customer attrition?


## Import Libraries
The analysis uses pandas and NumPy for data preparation, Matplotlib for visualization, and scikit-learn for preprocessing, modeling, validation, and evaluation.


In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


## Load Dataset
The preferred method is to place `WA_Fn-UseC_-Telco-Customer-Churn.csv` beside this notebook. A public raw mirror is included as a fallback for convenience.


In [ ]:
LOCAL_FILE = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv")
RAW_MIRROR = "https://raw.githubusercontent.com/SaeidRostami/Customer_Churn/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"

if LOCAL_FILE.exists():
    df = pd.read_csv(LOCAL_FILE)
    source_used = str(LOCAL_FILE)
else:
    df = pd.read_csv(RAW_MIRROR)
    source_used = RAW_MIRROR

print("Source:", source_used)
print("Shape:", df.shape)
df.head()


## Data Quality Audit
The raw dataset contains customer identifiers, demographics, service subscriptions, account information, charges, and the churn outcome. `customerID` is an identifier rather than a predictive business characteristic and is removed before modeling.


In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique()
})
display(audit)
print("Duplicate rows:", df.duplicated().sum())


## Data Cleaning
`TotalCharges` may contain blank strings, so it is converted to numeric. Rows with unusable total-charge values are treated as missing and handled explicitly. Duplicate rows are removed. The target is converted from Yes/No to 1/0.


In [ ]:
clean = df.copy()
clean["TotalCharges"] = pd.to_numeric(clean["TotalCharges"], errors="coerce")
before = len(clean)
clean = clean.drop_duplicates().reset_index(drop=True)
print("Duplicates removed:", before - len(clean))
print("Missing TotalCharges after conversion:", clean["TotalCharges"].isna().sum())

clean["ChurnFlag"] = clean["Churn"].map({"No": 0, "Yes": 1})
print("Rows:", len(clean))
print("Overall churn rate:", round(clean["ChurnFlag"].mean(), 4))


## Exploratory Data Analysis
The following plots examine overall churn and relationships with contract type, tenure, monthly charges, internet service, and payment method. These variables are business-relevant and can support interpretable retention recommendations.


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
counts = clean["Churn"].value_counts().reindex(["No","Yes"])
ax.bar(counts.index, counts.values)
ax.set_title("Customer Churn Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

contract_rate = clean.groupby("Contract")["ChurnFlag"].mean().sort_values(ascending=False)
display((contract_rate*100).round(1).rename("Churn Rate (%)"))
contract_rate.plot(kind="bar", figsize=(7,4), title="Churn Rate by Contract Type")
plt.ylabel("Churn Rate")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for label, group in clean.groupby("Churn"):
    ax.hist(group["tenure"], bins=20, alpha=0.55, label=label)
ax.set_title("Tenure Distribution by Churn")
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Customers")
ax.legend(title="Churn")
plt.tight_layout()
plt.show()

clean.boxplot(column="MonthlyCharges", by="Churn", figsize=(6,4))
plt.title("Monthly Charges by Churn")
plt.suptitle("")
plt.ylabel("Monthly Charges")
plt.tight_layout()
plt.show()


In [ ]:
internet_rate = (clean.groupby("InternetService")["ChurnFlag"].mean()*100).sort_values(ascending=False)
payment_rate = (clean.groupby("PaymentMethod")["ChurnFlag"].mean()*100).sort_values(ascending=False)
print("Churn rate by Internet Service (%)")
display(internet_rate.round(1))
print("Churn rate by Payment Method (%)")
display(payment_rate.round(1))


## Feature Engineering and Preprocessing
The identifier and original text target are excluded. Numeric features are median-imputed and standardized. Categorical features are most-frequent-imputed and one-hot encoded. Using a `ColumnTransformer` inside each model pipeline prevents preprocessing leakage from the test set.


In [ ]:
X = clean.drop(columns=["customerID", "Churn", "ChurnFlag"])
y = clean["ChurnFlag"]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train churn rate:", round(y_train.mean(),4), "Test churn rate:", round(y_test.mean(),4))


## Machine Learning Models
Three classifiers are compared:
- **Logistic Regression:** interpretable linear baseline with class balancing.
- **Decision Tree:** nonlinear and easy to explain, but prone to overfitting.
- **Random Forest:** ensemble model that can capture nonlinear interactions and typically generalizes better than a single tree.

The models use identical train/test splits and preprocessing for a fair comparison.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=20, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, min_samples_leaf=3, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    )
}

pipelines = {
    name: Pipeline([("prep", preprocessor), ("model", model)])
    for name, model in models.items()
}


## Cross-Validation
Stratified 5-fold cross-validation estimates out-of-sample performance while preserving the churn class proportion. ROC-AUC and F1 are emphasized because the classes are imbalanced and accuracy alone can hide poor detection of churners.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy":"accuracy", "precision":"precision", "recall":"recall",
           "f1":"f1", "roc_auc":"roc_auc"}

cv_rows = []
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"Model": name}
    for metric in scoring:
        row[f"CV {metric.title()} Mean"] = scores[f"test_{metric}"].mean()
        row[f"CV {metric.title()} SD"] = scores[f"test_{metric}"].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values("CV Roc_Auc Mean", ascending=False)
display(cv_results.round(4))


## Holdout Test Evaluation
Each model is fitted to the training set and evaluated once on the untouched test set. Precision measures how many predicted churners actually churned; recall measures how many actual churners were detected; F1 balances precision and recall; ROC-AUC evaluates ranking quality across thresholds.


In [ ]:
test_rows = []
fitted = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:,1]
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

test_results = pd.DataFrame(test_rows).sort_values("ROC-AUC", ascending=False)
display(test_results.round(4))


In [ ]:
best_name = test_results.iloc[0]["Model"]
best_pipe = fitted[best_name]
best_pred = best_pipe.predict(X_test)
best_prob = best_pipe.predict_proba(X_test)[:,1]

print("Selected model by holdout ROC-AUC:", best_name)
print(classification_report(y_test, best_pred, target_names=["No Churn","Churn"]))

ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, display_labels=["No Churn","Churn"]
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7,5))
for name, pipe in fitted.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)
ax.set_title("ROC Curves — Holdout Test Set")
plt.tight_layout()
plt.show()


## Model Interpretation
For Logistic Regression, transformed coefficients show the direction and strength of association with churn. For tree models, feature importances show which transformed variables contribute most to splitting decisions. These are model associations, not proof of causation.


In [ ]:
def transformed_feature_names(pipe):
    return pipe.named_steps["prep"].get_feature_names_out()

if "Logistic Regression" in fitted:
    log_pipe = fitted["Logistic Regression"]
    names = transformed_feature_names(log_pipe)
    coefs = log_pipe.named_steps["model"].coef_[0]
    coef_df = pd.DataFrame({"Feature": names, "Coefficient": coefs})
    coef_df["AbsCoefficient"] = coef_df["Coefficient"].abs()
    top_coef = coef_df.sort_values("AbsCoefficient", ascending=False).head(15)
    display(top_coef[["Feature","Coefficient"]].round(4))

    plot_df = top_coef.sort_values("Coefficient")
    plt.figure(figsize=(8,6))
    plt.barh(plot_df["Feature"], plot_df["Coefficient"])
    plt.title("Top Logistic Regression Coefficients")
    plt.xlabel("Coefficient")
    plt.tight_layout()
    plt.show()

rf_pipe = fitted["Random Forest"]
rf_names = transformed_feature_names(rf_pipe)
rf_imp = rf_pipe.named_steps["model"].feature_importances_
rf_df = pd.DataFrame({"Feature":rf_names, "Importance":rf_imp}).sort_values("Importance", ascending=False).head(15)
display(rf_df.round(4))
rf_df.sort_values("Importance").plot(kind="barh", x="Feature", y="Importance", legend=False, figsize=(8,6), title="Random Forest Feature Importance")
plt.tight_layout()
plt.show()


## Research Question Answers
The following cell creates evidence-based answers from the computed results. Review the EDA tables and model interpretation above before finalizing the narrative.


In [ ]:
best = test_results.iloc[0]
answers = {
    "RQ1": "Use the contract, tenure, monthly-charge, internet-service, payment-method, and model-interpretation outputs above to identify the strongest observed associations. Do not describe them as causal.",
    "RQ2": f"Yes. On the held-out test set, the selected {best['Model']} model achieved ROC-AUC={best['ROC-AUC']:.3f}, accuracy={best['Accuracy']:.3f}, precision={best['Precision']:.3f}, recall={best['Recall']:.3f}, and F1={best['F1']:.3f}.",
    "RQ3": f"{best['Model']} ranked highest by holdout ROC-AUC ({best['ROC-AUC']:.3f}). Compare this with the 5-fold cross-validation table before declaring it the final model.",
    "RQ4": "Retention actions should focus on high-risk segments identified by interpretable patterns, while using human review, avoiding discriminatory targeting, testing interventions experimentally, and monitoring model drift and unequal error rates."
}
for k,v in answers.items():
    print(k, v, "\n")


## Ethical Considerations
Customer-churn models can create real consequences even when the source data are de-identified. Key issues include:
- **Privacy and data minimization:** use only variables needed for the retention purpose and protect customer records.
- **Proxy discrimination:** demographic or service variables may correlate with protected characteristics. Predictions should not be used to deny service or impose worse terms.
- **Unequal error costs:** false negatives miss customers who may leave, while false positives may lead to unnecessary targeting. Error rates should be reviewed across relevant groups.
- **Transparency:** stakeholders should understand the model's purpose, limitations, evaluation metrics, and non-causal nature of feature associations.
- **Human oversight:** predictions should prioritize supportive retention outreach, not fully automated adverse decisions.
- **Data validity and drift:** a historical public sample may not represent a current telecom customer base. Production use requires local validation and ongoing monitoring.


## Conclusion
This project demonstrates an end-to-end churn-prediction workflow: data quality review, cleaning, exploratory analysis, leakage-safe preprocessing, model comparison, cross-validation, holdout evaluation, interpretation, and ethical review. The final model should be selected using multiple metrics rather than accuracy alone. The most useful outcome is not simply a churn label; it is a transparent risk signal that can help decision-makers prioritize retention efforts while preserving fairness, privacy, and human judgment.


## References
IBM. (n.d.). *Telco customer churn*. Kaggle. https://www.kaggle.com/datasets/blastchar/telco-customer-churn

Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

Sana, J. K., Abedin, M. Z., Rahman, M. S., & Rahman, M. S. (2022). Data transformation based optimized customer churn prediction model for the telecommunication industry. *arXiv*. https://arxiv.org/abs/2201.04088

Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*.
